In [ ]:
import numpy as np
from perf_func import time_stage, test_concurrency
import numba

1. numpy

In [2]:
# Pseudo-random sequence (TS 36.211 §7.2)
def c_sequence(M, c_init, Nc=1600):
    """Gold sequence of length M, seeded by c_init."""
    x_1 = np.zeros(M + Nc, dtype=np.uint8)
    x_1[0] = 1
    for n in range(31, M + Nc):
        x_1[n] = x_1[n - 28] ^ x_1[n - 31]

    x_2 = np.zeros(M + Nc, dtype=np.uint8)
    for n in range(31):
        x_2[n] = (c_init & (1 << n)) >> n
    for n in range(31, M + Nc):
        x_2[n] = x_2[n - 28] ^ x_2[n - 29] ^ x_2[n - 30] ^ x_2[n - 31]

    return x_1[Nc:] ^ x_2[Nc:]

# CRS sequence and location (TS 36.211 §6.10.1) 
def crs_seq(ns, l, N_id, N_cp=1, N_RB_MAX_DL = 110):
    """Cell-specific reference signal sequence."""
    c_init = (((7 * (ns + 1) + l + 1) * (2 * N_id + 1)) << 10) + 2 * N_id + N_cp
    c = c_sequence(4 * N_RB_MAX_DL, c_init)
    return np.sqrt(0.5) * ((1 - 2.0 * c[0::2]) + 1j * (1 - 2.0 * c[1::2]))


def extract_OFDM(ofdm_symbol, N_rb, N_rb_sc=12):
    """FFT + extract center N_rb*N_rb_sc subcarriers, skipping DC."""
    re = np.fft.fftshift(np.fft.fft(ofdm_symbol))
    N_sc = N_rb * N_rb_sc
    N_FFT = len(ofdm_symbol)
    active_sc = np.concatenate((
        np.arange(N_FFT // 2 - N_sc // 2, N_FFT // 2),
        np.arange(N_FFT // 2 + 1, N_FFT // 2 + N_sc // 2 + 1)
    ))
    return re[active_sc]

def crs_syms_and_k(N_rb, p, l, ns, N_id, N_cp=1, N_symb_DL=7, N_RB_MAX_DL = 110):
    """CRS subcarrier indices and values for antenna port p,
    symbol l, slot ns. Returns (k, crs) or (None, None)."""
    nu = -1
    if p == 0:
        if l == 0:       nu = 0
        elif l == N_symb_DL - 3: nu = 3
    if p == 1:
        if l == 0:       nu = 3
        elif l == N_symb_DL - 3: nu = 0
    if p == 2 and l == 1:
        nu = 3 * (ns % 2)
    if p == 3 and l == 1:
        nu = 3 + 3 * (ns % 2)

    if nu == -1:
        return None, None

    nu_shift = N_id % 6
    m = np.arange(2 * N_rb)
    k = 6 * m + (nu + nu_shift) % 6

    mm = m + N_RB_MAX_DL - N_rb
    rr = crs_seq(ns, l, N_id, N_cp)
    crs_syms = rr[mm]

    return k, crs_syms

In [3]:
class CRSChannelEstimation:

    def __init__(self, params):
        self.N_FFT = params.N_FFT
        self.N_CP = params.N_CP
        self.N_CP_extra = params.N_CP_extra
        self.Fs = params.Fs
        self._crs_table = None
        self._time_table = None
        self._n_symbols = None
        self._is_slot_start = None
        self._N_rb = None

    def config(self, N_id, N_rb, ns, n_symbols, is_slot_start=False):
        """Precompute CRS positions and interpolation indices for all 4 ports."""
        self._n_symbols = n_symbols
        self._is_slot_start = is_slot_start
        self._N_rb = N_rb
        N_sc = N_rb * 12

        # per port: list of (l, k, crs, freq_idx)
        self._crs_table = [[] for _ in range(4)]

        for p in range(4):
            for l in range(n_symbols):
                local_l = l % 7
                local_ns = ns + (l // 7)
                k, crs = crs_syms_and_k(N_rb, p, local_l, local_ns, N_id)
                if k is None:
                    continue
                freq_idx = np.array([np.argmin(np.abs(k - kk)) for kk in range(N_sc)])  
                self._crs_table[p].append((l, k, crs, freq_idx))

        # per port: list of (l, nearest_crs_l)
        self._time_table = [[] for _ in range(4)]

        for p in range(4):
            crs_syms = [entry[0] for entry in self._crs_table[p]]
            if not crs_syms:
                continue
            for l in range(n_symbols):
                if l not in crs_syms:
                    nearest = crs_syms[np.argmin([abs(l - s) for s in crs_syms])]
                    self._time_table[p].append((l, nearest))

    def __call__(self, chunk):
        corrected = self._freq_correct(chunk.data, chunk.f_d)
        self._fft_extract(corrected, chunk)
        self._estimate_h(chunk)
        self._interpolate_time(chunk)
        return chunk
    
    def _freq_correct(self, data, f_d):
        """Frequency offset correction in time domain."""
        return data * np.exp(-2j * np.pi * f_d * np.arange(len(data)) / self.Fs)

    def _fft_extract(self, corrected, chunk):
        """FFT each OFDM symbol, extract active subcarriers."""
        start = 0
        for l in range(self._n_symbols):
            cp = self.N_CP
            if l % 7 == 0 and self._is_slot_start:
                cp += self.N_CP_extra
            start += cp
            chunk.symbols[l, :] = extract_OFDM(corrected[start:start + self.N_FFT], self._N_rb)
            start += self.N_FFT

    def _estimate_h(self, chunk):
        """Estimate channel at CRS positions, interpolate in frequency."""
        for p in range(chunk.n_ant):
            for l, k, crs, freq_idx in self._crs_table[p]:
                h_crs = chunk.symbols[l, k] * np.conj(crs)
                chunk.H[p, l, :] = h_crs[freq_idx]

    def _interpolate_time(self, chunk):
        """Copy channel estimate from nearest CRS symbol to non-CRS symbols."""
        for p in range(chunk.n_ant):
            for l, nearest in self._time_table[p]:
                chunk.H[p, l, :] = chunk.H[p, nearest, :]

In [4]:
# ---- Equalization (TS 36.211 §6.3.4.3) ----

def equalize_1ant(r, H):
    return r / H[0]

def equalize_2ant(r, H):
    x = np.zeros_like(r)
    H0, H1 = H[0], H[1]
    scale = np.abs(H0[0::2])**2 + np.abs(H1[0::2])**2
    x[0::2] = (H0[0::2].conj() * r[0::2] + H1[0::2] * r[1::2].conj()) / scale
    x[1::2] = ((-H1[0::2].conj() * r[0::2] + H0[0::2] * r[1::2].conj()) / scale).conj()
    return x

def equalize_4ant(r, H):
    x = np.zeros_like(r)
    H0, H1, H2, H3 = H[0], H[1], H[2], H[3]
    scale02 = np.abs(H0[0::4])**2 + np.abs(H2[0::4])**2
    x[0::4] = (H0[0::4].conj() * r[0::4] + H2[0::4] * r[1::4].conj()) / scale02
    x[1::4] = ((-H2[0::4].conj() * r[0::4] + H0[0::4] * r[1::4].conj()) / scale02).conj()
    scale13 = np.abs(H1[2::4])**2 + np.abs(H3[2::4])**2
    x[2::4] = (H1[2::4].conj() * r[2::4] + H3[2::4] * r[3::4].conj()) / scale13
    x[3::4] = ((-H3[2::4].conj() * r[2::4] + H1[2::4] * r[3::4].conj()) / scale13).conj()
    return x

def descramble(pbch_bits, scramble_seq):
    n_bits = len(pbch_bits)
    best_sec, best_dist, best_bits = None, np.inf, None
    for s in range(4):
        candidate = pbch_bits ^ scramble_seq[s * n_bits:(s + 1) * n_bits]
        reps = candidate.reshape(4, 120)
        dist = 0
        for i in range(4):
            for j in range(i + 1, 4):
                dist += np.sum(reps[i] != reps[j])
        if dist < best_dist:
            best_dist, best_sec, best_bits = dist, s, candidate
    reps = best_bits.reshape(4, 120)
    pbch_bits = (np.sum(reps, axis=0) >= 2).astype(np.uint8)
    return best_sec, pbch_bits


col_perm_table = np.array([1, 17, 9, 25, 5, 21, 13, 29, 3, 19, 11, 27, 7, 23, 15, 31,
                            0, 16, 8, 24, 4, 20, 12, 28, 2, 18, 10, 26, 6, 22, 14, 30])

def subblock_interleaver(seq_len, col_perm_table=col_perm_table):
    N_cc = 32
    D = seq_len
    DUMMY = D + 10
    R = (D + N_cc - 1) // N_cc
    N_dummy = R * N_cc - D
    y = np.concatenate((DUMMY * np.ones(N_dummy, dtype=int), np.arange(seq_len)))
    M = np.reshape(y, (R, N_cc))
    P = np.zeros_like(M)
    for n in range(N_cc):
        P[:, n] = M[:, col_perm_table[n]]
    v = np.reshape(P.T, -1)
    return v[v != DUMMY]


def _count_ones(n):
    b = 0
    while n:
        b += n & 1
        n >>= 1
    return b

def _count_bits(n):
    b = 0
    while n:
        b += 1
        n >>= 1
    return b

def hamming_dist(obs, ref):
    return np.sum(ref != obs)


class CRC16:
    def __init__(self, poly=0x1021):
        self._t = np.zeros(256, dtype=np.uint16)
        mask = np.uint16(1 << 15)
        for n in np.arange(256, dtype=np.uint16):
            c = n << 8
            for _ in range(8):
                c = (poly ^ (c << 1)) if (c & mask) else (c << 1)
            self._t[n] = c

    def __call__(self, data):
        crc = np.uint16(0)
        for d in data:
            crc = self._t[((crc >> 8) ^ d) & 0xFF] ^ ((crc << 8) & 0xFFFF)
        return crc & 0xFFFF


class ConvCoder:
    def __init__(self, generators):
        self.n_codes = len(generators)
        self.order = max(_count_bits(g) for g in generators) - 1
        self.Ns = 2 << (self.order - 1)
        self.Nt = 2 << self.order
        self._t = np.empty((self.Nt, self.n_codes), dtype=np.uint8)
        for n in range(self.Nt):
            for m in range(self.n_codes):
                self._t[n, m] = _count_ones(n & generators[m]) & 0x1

    def decode(self, d, cost_fun):
        N = d.shape[1]
        costs = np.zeros(self.Ns)
        survivors = np.zeros((self.Ns, N), dtype=np.uint8)
        for n in range(N):
            tmp_cost = np.inf * np.ones(self.Ns)
            tmp_b = np.zeros(self.Ns, dtype=np.uint8)
            tmp_survivors = survivors[:, :n].copy()
            obs = d[:, n]
            for te in np.arange(self.Ns, dtype=np.uint8):
                for b in np.arange(2, dtype=np.uint8):
                    t = (te << 1) + b
                    ts = t & (self.Ns - 1)
                    c = costs[ts] + cost_fun(obs, self._t[t, :])
                    if c < tmp_cost[te]:
                        tmp_cost[te] = c
                        tmp_b[te] = (t & self.Ns) >> self.order
                        tmp_survivors[te, :] = survivors[ts, :n]
            for te in np.arange(self.Ns, dtype=np.uint8):
                costs[te] = tmp_cost[te]
                survivors[te, :n] = tmp_survivors[te, :]
                survivors[te, n] = tmp_b[te]
        return survivors[np.argmin(costs), :], np.min(costs)

In [5]:
class PBCHDecoding:

    def __init__(self):
        self.EQUALIZE = {1: equalize_1ant, 2: equalize_2ant, 4: equalize_4ant}
        self._l_idx = None
        self._k_idx = None

    def config(self, N_id):
        self._l_idx, self._k_idx = self.pbch_data_mask(N_id)

    def __call__(self, chunk):
        r = chunk.symbols[self._l_idx, self._k_idx]
        H = chunk.H[:, self._l_idx, self._k_idx]
        chunk.pbch_eq = self.EQUALIZE[chunk.n_ant](r, H)
        chunk.pbch_bits[0::2] = (chunk.pbch_eq.real < 0).astype(np.uint8)
        chunk.pbch_bits[1::2] = (chunk.pbch_eq.imag < 0).astype(np.uint8)
        return chunk

    def pbch_data_mask(self, N_id, N_rb=6, n_symbols=4, N_rb_sc=12):
        N_sc = N_rb * N_rb_sc
        nu_shift = N_id % 6
        crs_offsets = {nu_shift, (nu_shift + 3) % 6}
        l_idx, k_idx = [], []
        for l in range(n_symbols):
            for kk in range(N_sc):
                if l < 2 and (kk % 6) in crs_offsets:
                    continue
                l_idx.append(l)
                k_idx.append(kk)
        return np.array(l_idx), np.array(k_idx)

In [6]:
class BCHDecoding:

    def __init__(self):
        self.BW_TABLE = {0: 6, 1: 15, 2: 25, 3: 50, 4: 75, 5: 100}
        self.PHICH_RES_TABLE = ['1/6', '1/2', '1', '2']
        self.ANT_MASKS = {1: 0x00, 2: 0xFF, 4: 0x33}
        self.perm_table = subblock_interleaver(40)
        self.fec = ConvCoder([0o133, 0o171, 0o165])
        self.crc = CRC16()
        self._scramble_seq = None

    def config(self, N_id):
        self._scramble_seq = c_sequence(1920, N_id)

    def __call__(self, chunk):
        sec, pbch_bits = descramble(chunk.pbch_bits, self._scramble_seq)

        coded = np.zeros((3, 40), dtype=np.uint8)
        for n in range(3):
            coded[n, self.perm_table] = pbch_bits[n * 40:(n + 1) * 40]

        coded_2x = np.concatenate((coded, coded), axis=1)
        decoded_2x, cost = self.fec.decode(coded_2x, hamming_dist)
        mib_bits = np.concatenate((decoded_2x[40:60], decoded_2x[20:40]))

        mib_bytes = np.packbits(mib_bits)
        mask = self.ANT_MASKS[chunk.n_ant]
        mib_bytes[3] ^= mask
        mib_bytes[4] ^= mask

        if self.crc(mib_bytes) != 0:
            return chunk

        chunk.cost = cost
        chunk.mib_decoded = True
        chunk.sec = sec
        chunk.dl_bw, chunk.phich_dur, chunk.phich_res, chunk.sfn = \
            self.parse_mib(mib_bytes[:3], sec)
        return chunk

    def parse_mib(self, mib_bytes, sec):
        bits = np.unpackbits(mib_bytes)
        bw_idx = int((bits[0] << 2) | (bits[1] << 1) | bits[2])
        dl_bw = self.BW_TABLE[bw_idx]
        phich_dur = 'extended' if bits[3] else 'normal'
        phich_res = self.PHICH_RES_TABLE[int((bits[4] << 1) | bits[5])]
        sfn_mib = 0
        for i in range(8):
            sfn_mib = (sfn_mib << 1) | bits[6 + i]
        sfn = int(sfn_mib) * 4 + sec
        return dl_bw, phich_dur, phich_res, sfn

In [7]:
class MIBChunk:

    def __init__(self, data, tag, N_id, f_d, N_rb=6, ns=1,
                 n_symbols=4, is_slot_start=True):
        N_sc = N_rb * 12

        # input
        self.data = data
        self.tag = tag
        self.N_id = N_id
        self.N_rb = N_rb
        self.N_sc = N_sc
        self.ns = ns
        self.f_d = f_d
        self.n_symbols = n_symbols
        self.is_slot_start = is_slot_start
        self.n_ant = 4

        # CRS outputs
        self.symbols = np.zeros((n_symbols, N_sc), dtype=complex)
        self.H = np.zeros((4, n_symbols, N_sc), dtype=complex)

        # PBCH outputs
        self.pbch_eq = np.zeros(240, dtype=complex)
        self.pbch_bits = np.zeros(480, dtype=np.uint8)

        # BCH outputs
        self.mib_decoded = False
        self.sec = None
        self.dl_bw = None
        self.phich_dur = None
        self.phich_res = None
        self.sfn = None
        self.cost = None

accuracy + time cost for numpy

In [8]:
from data.lte_system_info import LTEParams

In [9]:
rxf = np.load('data/rx_preprocessed.npy')
Fs = float(np.load('data/Fs.npy'))
params = LTEParams(Fs=Fs)

In [10]:
from cell_search import PSSDetection, SSSDetection, PSSChunk

In [11]:
N_overlap = params.N_CP + 2 * params.N_FFT
N_subframe = params.N_subframe
stride = N_subframe - N_overlap

pss = PSSDetection(params, peak_ratio=5.0)

pos = 0
chunk_id = 0
pss_count = 0
cells = []

while pos + N_subframe <= len(rxf):

    data = rxf[pos:pos+N_subframe]
    chunk = PSSChunk(data, chunk_id)
    chunk = pss(chunk)

    if chunk.pss_detected:
        pss_count += 1
        global_pss = pos + chunk.pss_local_index
        print(f"Chunk {chunk_id}: PSS detected | "
              f"N_id_2={chunk.N_id_2}, "
              f"local={chunk.pss_local_index}, global={global_pss}")
        cells.append(chunk)    

    pos += stride
    chunk_id += 1

print(f"\nScanned {chunk_id} chunks over {len(rxf)} samples, "
      f"{pss_count} PSS detected")

Chunk 0: PSS detected | N_id_2=2, local=12309, global=12309
Chunk 2: PSS detected | N_id_2=2, local=9563, global=36043
Chunk 6: PSS detected | N_id_2=2, local=9667, global=89107
Chunk 8: PSS detected | N_id_2=2, local=6922, global=112842
Chunk 12: PSS detected | N_id_2=2, local=7027, global=165907
Chunk 13: PSS detected | N_id_2=0, local=11192, global=183312
Chunk 14: PSS detected | N_id_2=2, local=4283, global=189643
Chunk 18: PSS detected | N_id_2=2, local=4390, global=242710
Chunk 19: PSS detected | N_id_2=2, local=14327, global=265887
Chunk 20: PSS detected | N_id_2=2, local=1642, global=266442
Chunk 22: PSS detected | N_id_2=2, local=14330, global=305610
Chunk 24: PSS detected | N_id_2=2, local=1746, global=319506
Chunk 25: PSS detected | N_id_2=2, local=12243, global=343243
Chunk 26: PSS detected | N_id_2=1, local=2587, global=346827
Chunk 29: PSS detected | N_id_2=2, local=12349, global=396309
Chunk 31: PSS detected | N_id_2=2, local=9602, global=420042
Chunk 35: PSS detected | 

In [12]:
sss = SSSDetection(params, peak_ratio=8.0)

sss_cells = []
for chunk in cells:
    chunk = sss(chunk)
    if chunk.sss_detected:
        global_pss = chunk.tag * stride + chunk.pss_local_index
        PCI = 3 * chunk.N_id_1 + chunk.N_id_2
        print(f"Chunk {chunk.tag}: PCI={PCI} | "
              f"N_id_1={chunk.N_id_1}, N_id_2={chunk.N_id_2}, F={chunk.F}, "
              f"local={chunk.pss_local_index}, global={global_pss} | "
              f"f_d={chunk.f_d:.1f}")
        sss_cells.append(chunk)

print(f"\n{len(sss_cells)} SSS detected out of {len(cells)} PSS chunks")

if sss_cells:
    first_global = sss_cells[0].tag * stride + sss_cells[0].pss_local_index
    expected = (len(rxf) - first_global) // params.N_half_frame + 1
    print(f"Expected {expected} cell found ({params.N_half_frame} spacing from {first_global})")

Chunk 2: PCI=380 | N_id_1=126, N_id_2=2, F=0, local=9563, global=36043 | f_d=1126.9
Chunk 8: PCI=380 | N_id_1=126, N_id_2=2, F=1, local=6922, global=112842 | f_d=1128.4
Chunk 14: PCI=380 | N_id_1=126, N_id_2=2, F=0, local=4283, global=189643 | f_d=1164.2
Chunk 20: PCI=380 | N_id_1=126, N_id_2=2, F=1, local=1642, global=266442 | f_d=1204.1
Chunk 25: PCI=380 | N_id_1=126, N_id_2=2, F=0, local=12243, global=343243 | f_d=1180.9
Chunk 31: PCI=380 | N_id_1=126, N_id_2=2, F=1, local=9602, global=420042 | f_d=1107.1
Chunk 37: PCI=380 | N_id_1=126, N_id_2=2, F=0, local=6962, global=496842 | f_d=1179.5
Chunk 43: PCI=380 | N_id_1=126, N_id_2=2, F=1, local=4323, global=573643 | f_d=1142.0
Chunk 49: PCI=380 | N_id_1=126, N_id_2=2, F=0, local=1681, global=650441 | f_d=1269.2
Chunk 54: PCI=380 | N_id_1=126, N_id_2=2, F=1, local=12282, global=727242 | f_d=1165.1
Chunk 60: PCI=380 | N_id_1=126, N_id_2=2, F=0, local=9642, global=804042 | f_d=1122.2
Chunk 66: PCI=380 | N_id_1=126, N_id_2=2, F=1, local=70

In [13]:
crs_np = CRSChannelEstimation(params)
pbch_np = PBCHDecoding()
bch_np = BCHDecoding()

N_id = 3 * sss_cells[0].N_id_1 + sss_cells[0].N_id_2

crs_np.config(N_id=N_id, N_rb=6, ns=1, n_symbols=4, is_slot_start=True)
pbch_np.config(N_id=N_id)
bch_np.config(N_id=N_id)

np_mib_cells = []
for cell in sss_cells:
    if cell.F != 0:
        continue
    f_d = cell.f_d
    pss_global = cell.tag * stride + cell.pss_local_index
    slot1_start = pss_global + params.N_FFT
    data = rxf[slot1_start:slot1_start + params.pbch_len]

    chunk = MIBChunk(data, tag=cell.tag, N_id=N_id, f_d=f_d)
    chunk.pss_global = pss_global  # add this for later reference
    crs_np(chunk)
    for n_ant in [1, 2, 4]:
        chunk.n_ant = n_ant
        pbch_np(chunk)
        bch_np(chunk)
        if chunk.mib_decoded:
            break

    np_mib_cells.append(chunk)
    if chunk.mib_decoded:
        print(f"Chunk {cell.tag}: SFN={chunk.sfn}, BW={chunk.dl_bw}, n_ant={chunk.n_ant}, sec={chunk.sec}, cost={chunk.cost}")
    else:
        print(f"Chunk {cell.tag}: FAILED")

print(f"\n{sum(c.mib_decoded for c in np_mib_cells)}/{len(np_mib_cells)} decoded")

Chunk 2: SFN=313, BW=50, n_ant=2, sec=1, cost=0.0
Chunk 14: SFN=314, BW=50, n_ant=2, sec=2, cost=0.0
Chunk 25: SFN=315, BW=50, n_ant=2, sec=3, cost=0.0
Chunk 37: SFN=316, BW=50, n_ant=2, sec=0, cost=0.0
Chunk 49: SFN=317, BW=50, n_ant=2, sec=1, cost=0.0
Chunk 60: SFN=318, BW=50, n_ant=2, sec=2, cost=0.0
Chunk 72: SFN=319, BW=50, n_ant=2, sec=3, cost=0.0
Chunk 83: SFN=320, BW=50, n_ant=2, sec=0, cost=0.0
Chunk 95: SFN=321, BW=50, n_ant=2, sec=1, cost=0.0
Chunk 107: SFN=322, BW=50, n_ant=2, sec=2, cost=0.0
Chunk 118: SFN=323, BW=50, n_ant=2, sec=3, cost=0.0
Chunk 130: SFN=324, BW=50, n_ant=2, sec=0, cost=0.0
Chunk 141: SFN=325, BW=50, n_ant=2, sec=1, cost=0.0
Chunk 153: SFN=326, BW=50, n_ant=2, sec=2, cost=0.0
Chunk 165: SFN=327, BW=50, n_ant=2, sec=3, cost=0.0
Chunk 176: SFN=328, BW=50, n_ant=2, sec=0, cost=0.0
Chunk 188: SFN=329, BW=50, n_ant=2, sec=1, cost=0.0
Chunk 199: SFN=330, BW=50, n_ant=2, sec=2, cost=0.0
Chunk 211: SFN=331, BW=50, n_ant=2, sec=3, cost=0.0
Chunk 223: SFN=332, BW

In [14]:
# 1. unit time
crs_time = time_stage(crs_np, np_mib_cells[0])
print(f"CRS unit time: {crs_time:.2f} ms\n")

# 2. concurrency
results = test_concurrency(crs_np, np_mib_cells)

print(f"CRS concurrency test: {len(np_mib_cells)} chunks\n")
print(f"{'workers':>8s}  {'cc':>4s}  {'unit(ms)':>10s}  {'speedup':>8s}")
print("-" * 37)
baseline = results[0][2]
n = len(np_mib_cells)
for workers, cc, elapsed in results:
    unit_ms = elapsed / n * 1000
    print(f"{workers:>8d}  {cc:>4d}  {unit_ms:>10.2f}  {baseline/elapsed:>7.2f}x")

CRS unit time: 0.24 ms

CRS concurrency test: 100 chunks

 workers    cc    unit(ms)   speedup
-------------------------------------
       1     1        0.24     1.00x
       2     2        0.17     1.39x
       4     4        0.24     0.97x
       6     6        0.27     0.87x
      10    10        0.37     0.65x


In [15]:
# 1. unit time
pbch_time = time_stage(pbch_np, np_mib_cells[0])
print(f"PBCH unit time: {pbch_time:.2f} ms\n")

# 2. concurrency
results = test_concurrency(pbch_np, np_mib_cells)

print(f"PBCH concurrency test: {len(np_mib_cells)} chunks\n")
print(f"{'workers':>8s}  {'cc':>4s}  {'unit(ms)':>10s}  {'speedup':>8s}")
print("-" * 37)
baseline = results[0][2]
n = len(np_mib_cells)
for workers, cc, elapsed in results:
    unit_ms = elapsed / n * 1000
    print(f"{workers:>8d}  {cc:>4d}  {unit_ms:>10.2f}  {baseline/elapsed:>7.2f}x")

PBCH unit time: 0.02 ms

PBCH concurrency test: 100 chunks

 workers    cc    unit(ms)   speedup
-------------------------------------
       1     1        0.07     1.00x
       2     2        0.08     0.84x
       4     4        0.09     0.78x
       6     6        0.12     0.57x
      10    10        0.12     0.56x


In [16]:
# 1. unit time
bch_time = time_stage(bch_np, np_mib_cells[0])
print(f"BCH unit time: {bch_time:.2f} ms\n")

# 2. concurrency
results = test_concurrency(bch_np, np_mib_cells)

print(f"BCH concurrency test: {len(np_mib_cells)} chunks\n")
print(f"{'workers':>8s}  {'cc':>4s}  {'unit(ms)':>10s}  {'speedup':>8s}")
print("-" * 37)
baseline = results[0][2]
n = len(np_mib_cells)
for workers, cc, elapsed in results:
    unit_ms = elapsed / n * 1000
    print(f"{workers:>8d}  {cc:>4d}  {unit_ms:>10.2f}  {baseline/elapsed:>7.2f}x")

BCH unit time: 47.06 ms

BCH concurrency test: 100 chunks

 workers    cc    unit(ms)   speedup
-------------------------------------
       1     1       43.18     1.00x
       2     2       41.66     1.04x
       4     4       42.12     1.03x
       6     6       43.20     1.00x
      10    10       43.54     0.99x


2. numba

In [17]:
@numba.jit(nopython=True, nogil=True)
def _pre_fft_nb(data, f_d, Fs, sym_starts, N_FFT, n_symbols):
    """Freq correct + stack OFDM symbols. One Numba call, zero temp allocation."""
    phase_inc = -2.0 * np.pi * f_d / Fs
    ofdm = np.empty((n_symbols, N_FFT), dtype=np.complex128)
    for l in range(n_symbols):
        start = sym_starts[l]
        for i in range(N_FFT):
            idx = start + i
            phase = phase_inc * idx
            ofdm[l, i] = data[idx] * (np.cos(phase) + 1j * np.sin(phase))
    return ofdm


@numba.jit(nopython=True, nogil=True)
def _post_fft_nb(fft_all, fft_indices, symbols, H, n_ant,
                 n_crs_entries, crs_port, crs_l, crs_k, crs_conj, crs_freq_idx,
                 n_crs_per, N_sc,
                 n_time_entries, time_port, time_l, time_nearest):
    """Extract subcarriers + channel estimation + time interpolation. One Numba call."""
    n_symbols = fft_all.shape[0]

    # ---- Extract active subcarriers ----
    for l in range(n_symbols):
        for i in range(N_sc):
            symbols[l, i] = fft_all[l, fft_indices[i]]

    # ---- Channel estimation at CRS positions ----
    for e in range(n_crs_entries):
        p = crs_port[e]
        if p >= n_ant:
            continue
        l = crs_l[e]

        h_crs = np.empty(n_crs_per, dtype=np.complex128)
        for i in range(n_crs_per):
            h_crs[i] = symbols[l, crs_k[e, i]] * crs_conj[e, i]

        for i in range(N_sc):
            H[p, l, i] = h_crs[crs_freq_idx[e, i]]

    # ---- Time interpolation ----
    for e in range(n_time_entries):
        p = time_port[e]
        if p >= n_ant:
            continue
        l = time_l[e]
        nearest = time_nearest[e]
        for i in range(N_sc):
            H[p, l, i] = H[p, nearest, i]

In [18]:
class CRSChannelEstimationNumba:

    def __init__(self, params):
        self.N_FFT = params.N_FFT
        self.N_CP = params.N_CP
        self.N_CP_extra = params.N_CP_extra
        self.Fs = params.Fs

        self._n_symbols = None
        self._N_sc = None
        self._sym_starts = None
        self._fft_indices = None
        self._n_crs_per = None

        self._n_crs_entries = 0
        self._crs_port = None
        self._crs_l = None
        self._crs_k = None
        self._crs_conj = None
        self._crs_freq_idx = None

        self._n_time_entries = 0
        self._time_port = None
        self._time_l = None
        self._time_nearest = None

    def config(self, N_id, N_rb, ns, n_symbols, is_slot_start=False):
        self._n_symbols = n_symbols
        N_sc = N_rb * 12
        self._N_sc = N_sc

        # symbol start offsets
        self._sym_starts = np.empty(n_symbols, dtype=np.int64)
        start = 0
        for l in range(n_symbols):
            cp = self.N_CP
            if l % 7 == 0 and is_slot_start:
                cp += self.N_CP_extra
            start += cp
            self._sym_starts[l] = start
            start += self.N_FFT

        # FFT extraction indices (raw FFT, no fftshift)
        N_FFT = self.N_FFT
        active_shifted = np.concatenate((
            np.arange(N_FFT // 2 - N_sc // 2, N_FFT // 2),
            np.arange(N_FFT // 2 + 1, N_FFT // 2 + N_sc // 2 + 1)
        ))
        self._fft_indices = ((active_shifted - N_FFT // 2) % N_FFT).astype(np.int64)

        # build CRS table
        crs_table = [[] for _ in range(4)]
        for p in range(4):
            for l in range(n_symbols):
                local_l = l % 7
                local_ns = ns + (l // 7)
                k, crs = crs_syms_and_k(N_rb, p, local_l, local_ns, N_id)
                if k is None:
                    continue
                freq_idx = np.array([np.argmin(np.abs(k - kk)) for kk in range(N_sc)])
                crs_table[p].append((l, k, crs, freq_idx))

        # flatten CRS table
        total_entries = sum(len(e) for e in crs_table)
        n_crs_per = 2 * N_rb
        self._n_crs_per = n_crs_per

        self._crs_port = np.empty(total_entries, dtype=np.int64)
        self._crs_l = np.empty(total_entries, dtype=np.int64)
        self._crs_k = np.empty((total_entries, n_crs_per), dtype=np.int64)
        self._crs_conj = np.empty((total_entries, n_crs_per), dtype=np.complex128)
        self._crs_freq_idx = np.empty((total_entries, N_sc), dtype=np.int64)

        idx = 0
        for p in range(4):
            for l, k, crs, freq_idx in crs_table[p]:
                self._crs_port[idx] = p
                self._crs_l[idx] = l
                self._crs_k[idx, :] = k
                self._crs_conj[idx, :] = np.conj(crs)
                self._crs_freq_idx[idx, :] = freq_idx
                idx += 1
        self._n_crs_entries = idx

        # flatten time interpolation table
        time_entries = []
        for p in range(4):
            crs_syms = [entry[0] for entry in crs_table[p]]
            if not crs_syms:
                continue
            for l in range(n_symbols):
                if l not in crs_syms:
                    nearest = crs_syms[np.argmin([abs(l - s) for s in crs_syms])]
                    time_entries.append((p, l, nearest))

        n_time = len(time_entries)
        self._time_port = np.empty(n_time, dtype=np.int64)
        self._time_l = np.empty(n_time, dtype=np.int64)
        self._time_nearest = np.empty(n_time, dtype=np.int64)
        for i, (p, l, nearest) in enumerate(time_entries):
            self._time_port[i] = p
            self._time_l[i] = l
            self._time_nearest[i] = nearest
        self._n_time_entries = n_time

    def __call__(self, chunk):
        # 1. Freq correct + stack symbols (one Numba call, nogil)
        ofdm_stack = _pre_fft_nb(chunk.data, chunk.f_d, self.Fs,
                                  self._sym_starts, self.N_FFT, self._n_symbols)

        # 2. Batch FFT (one NumPy call, releases GIL)
        fft_all = np.fft.fft(ofdm_stack, axis=1)

        # 3. Extract + estimate + interpolate (one Numba call, nogil)
        _post_fft_nb(fft_all, self._fft_indices, chunk.symbols, chunk.H, chunk.n_ant,
                     self._n_crs_entries,
                     self._crs_port, self._crs_l, self._crs_k,
                     self._crs_conj, self._crs_freq_idx,
                     self._n_crs_per, self._N_sc,
                     self._n_time_entries,
                     self._time_port, self._time_l, self._time_nearest)

        return chunk

In [19]:
# ---- Numba Viterbi ----

@numba.jit(nopython=True, nogil=True)
def _viterbi_decode_numba(d, transition_table, order, Ns):
    """Viterbi decode with forward ACS + backward traceback. GIL released.

    d:                (n_codes, N) uint8 — observed coded bits
    transition_table: (Nt, n_codes) uint8 — encoder output for each transition
    order:            int — constraint length - 1
    Ns:               int — number of states (2^order)
    """
    n_codes = d.shape[0]
    N = d.shape[1]

    # Forward: ACS
    costs = np.zeros(Ns, dtype=np.float64)
    decisions = np.zeros((N, Ns), dtype=np.uint8)

    for n in range(N):
        new_costs = np.full(Ns, 1e30)

        for te in range(Ns):
            for b in range(2):
                t = (te << 1) + b
                ts = t & (Ns - 1)

                # inline hamming distance
                dist = 0
                for m in range(n_codes):
                    if d[m, n] != transition_table[t, m]:
                        dist += 1

                c = costs[ts] + dist
                if c < new_costs[te]:
                    new_costs[te] = c
                    decisions[n, te] = b

        for te in range(Ns):
            costs[te] = new_costs[te]

    # Find best final state
    best_state = 0
    best_cost = costs[0]
    for s in range(1, Ns):
        if costs[s] < best_cost:
            best_cost = costs[s]
            best_state = s

    # Backward: traceback
    decoded = np.zeros(N, dtype=np.uint8)
    state = best_state
    for n in range(N - 1, -1, -1):
        b = decisions[n, state]
        t = (state << 1) + b
        decoded[n] = (t & Ns) >> order
        state = t & (Ns - 1)

    return decoded, best_cost


class ConvCoderNumba:

    def __init__(self, generators):
        self.n_codes = len(generators)
        self.order = max(_count_bits(g) for g in generators) - 1
        self.Ns = 2 << (self.order - 1)
        self.Nt = 2 << self.order
        self._t = np.empty((self.Nt, self.n_codes), dtype=np.uint8)
        for n in range(self.Nt):
            for m in range(self.n_codes):
                self._t[n, m] = _count_ones(n & generators[m]) & 0x1

        # warmup numba
        dummy = np.zeros((self.n_codes, 10), dtype=np.uint8)
        _viterbi_decode_numba(dummy, self._t, self.order, self.Ns)

    def decode(self, d, cost_fun=None):
        return _viterbi_decode_numba(d, self._t, self.order, self.Ns)

In [20]:
class BCHDecodingNumba:

    def __init__(self):
        self.BW_TABLE = {0: 6, 1: 15, 2: 25, 3: 50, 4: 75, 5: 100}
        self.PHICH_RES_TABLE = ['1/6', '1/2', '1', '2']
        self.ANT_MASKS = {1: 0x00, 2: 0xFF, 4: 0x33}
        self.perm_table = subblock_interleaver(40)
        self.fec = ConvCoderNumba([0o133, 0o171, 0o165])
        self.crc = CRC16()
        self._scramble_seq = None

    def config(self, N_id):
        self._scramble_seq = c_sequence(1920, N_id)

    def __call__(self, chunk):
        sec, pbch_bits = descramble(chunk.pbch_bits, self._scramble_seq)

        coded = np.zeros((3, 40), dtype=np.uint8)
        for n in range(3):
            coded[n, self.perm_table] = pbch_bits[n * 40:(n + 1) * 40]

        coded_2x = np.concatenate((coded, coded), axis=1)
        decoded_2x, cost = self.fec.decode(coded_2x, None)
        mib_bits = np.concatenate((decoded_2x[40:60], decoded_2x[20:40]))

        mib_bytes = np.packbits(mib_bits)
        mask = self.ANT_MASKS[chunk.n_ant]
        mib_bytes[3] ^= mask
        mib_bytes[4] ^= mask

        if self.crc(mib_bytes) != 0:
            return chunk

        chunk.cost = cost
        chunk.mib_decoded = True
        chunk.sec = sec
        chunk.dl_bw, chunk.phich_dur, chunk.phich_res, chunk.sfn = \
            self.parse_mib(mib_bytes[:3], sec)
        return chunk

    def parse_mib(self, mib_bytes, sec):
        bits = np.unpackbits(mib_bytes)
        bw_idx = int((bits[0] << 2) | (bits[1] << 1) | bits[2])
        dl_bw = self.BW_TABLE[bw_idx]
        phich_dur = 'extended' if bits[3] else 'normal'
        phich_res = self.PHICH_RES_TABLE[int((bits[4] << 1) | bits[5])]
        sfn_mib = 0
        for i in range(8):
            sfn_mib = (sfn_mib << 1) | bits[6 + i]
        sfn = int(sfn_mib) * 4 + sec
        return dl_bw, phich_dur, phich_res, sfn

In [21]:
crs_nb = CRSChannelEstimationNumba(params)
pbch_nb = PBCHDecoding()
bch_nb = BCHDecodingNumba()

N_id = 3 * sss_cells[0].N_id_1 + sss_cells[0].N_id_2

crs_nb.config(N_id=N_id, N_rb=6, ns=1, n_symbols=4, is_slot_start=True)
pbch_nb.config(N_id=N_id)
bch_nb.config(N_id=N_id)

nb_mib_cells = []
for cell in sss_cells:
    if cell.F != 0:
        continue
    f_d = cell.f_d
    pss_global = cell.tag * stride + cell.pss_local_index
    slot1_start = pss_global + params.N_FFT
    data = rxf[slot1_start:slot1_start + params.pbch_len]

    chunk = MIBChunk(data, tag=cell.tag, N_id=N_id, f_d=f_d)
    chunk.pss_global = pss_global  # add this for later reference
    crs_nb(chunk)
    for n_ant in [1, 2, 4]:
        chunk.n_ant = n_ant
        pbch_nb(chunk)
        bch_nb(chunk)
        if chunk.mib_decoded:
            break

    nb_mib_cells.append(chunk)
    if chunk.mib_decoded:
        print(f"Chunk {cell.tag}: SFN={chunk.sfn}, BW={chunk.dl_bw}, n_ant={chunk.n_ant}, cost={chunk.cost}")
    else:
        print(f"Chunk {cell.tag}: FAILED")

print(f"{sum(c.mib_decoded for c in nb_mib_cells)}/{len(nb_mib_cells)} decoded")

Chunk 2: SFN=313, BW=50, n_ant=2, cost=0.0
Chunk 14: SFN=314, BW=50, n_ant=2, cost=0.0
Chunk 25: SFN=315, BW=50, n_ant=2, cost=0.0
Chunk 37: SFN=316, BW=50, n_ant=2, cost=0.0
Chunk 49: SFN=317, BW=50, n_ant=2, cost=0.0
Chunk 60: SFN=318, BW=50, n_ant=2, cost=0.0
Chunk 72: SFN=319, BW=50, n_ant=2, cost=0.0
Chunk 83: SFN=320, BW=50, n_ant=2, cost=0.0
Chunk 95: SFN=321, BW=50, n_ant=2, cost=0.0
Chunk 107: SFN=322, BW=50, n_ant=2, cost=0.0
Chunk 118: SFN=323, BW=50, n_ant=2, cost=0.0
Chunk 130: SFN=324, BW=50, n_ant=2, cost=0.0
Chunk 141: SFN=325, BW=50, n_ant=2, cost=0.0
Chunk 153: SFN=326, BW=50, n_ant=2, cost=0.0
Chunk 165: SFN=327, BW=50, n_ant=2, cost=0.0
Chunk 176: SFN=328, BW=50, n_ant=2, cost=0.0
Chunk 188: SFN=329, BW=50, n_ant=2, cost=0.0
Chunk 199: SFN=330, BW=50, n_ant=2, cost=0.0
Chunk 211: SFN=331, BW=50, n_ant=2, cost=0.0
Chunk 223: SFN=332, BW=50, n_ant=2, cost=0.0
Chunk 234: SFN=333, BW=50, n_ant=2, cost=0.0
Chunk 246: SFN=334, BW=50, n_ant=2, cost=0.0
Chunk 257: SFN=335, 

In [22]:
# 1. unit time
crs_time = time_stage(crs_nb, nb_mib_cells[0])
print(f"CRS unit time: {crs_time:.2f} ms\n")

# 2. concurrency
results = test_concurrency(crs_nb, nb_mib_cells)

print(f"CRS concurrency test: {len(nb_mib_cells)} chunks\n")
print(f"{'workers':>8s}  {'cc':>4s}  {'unit(ms)':>10s}  {'speedup':>8s}")
print("-" * 37)
baseline = results[0][2]
n = len(nb_mib_cells)
for workers, cc, elapsed in results:
    unit_ms = elapsed / n * 1000
    print(f"{workers:>8d}  {cc:>4d}  {unit_ms:>10.2f}  {baseline/elapsed:>7.2f}x")

CRS unit time: 0.06 ms

CRS concurrency test: 100 chunks

 workers    cc    unit(ms)   speedup
-------------------------------------
       1     1        0.08     1.00x
       2     2        0.06     1.46x
       4     4        0.07     1.10x
       6     6        0.12     0.67x
      10    10        0.12     0.67x


In [23]:
# 1. unit time
pbch_time = time_stage(pbch_nb, nb_mib_cells[0])
print(f"PBCH unit time: {pbch_time:.2f} ms\n")

# 2. concurrency
results = test_concurrency(pbch_nb, nb_mib_cells)

print(f"PBCH concurrency test: {len(nb_mib_cells)} chunks\n")
print(f"{'workers':>8s}  {'cc':>4s}  {'unit(ms)':>10s}  {'speedup':>8s}")
print("-" * 37)
baseline = results[0][2]
n = len(nb_mib_cells)
for workers, cc, elapsed in results:
    unit_ms = elapsed / n * 1000
    print(f"{workers:>8d}  {cc:>4d}  {unit_ms:>10.2f}  {baseline/elapsed:>7.2f}x")  

PBCH unit time: 0.03 ms

PBCH concurrency test: 100 chunks

 workers    cc    unit(ms)   speedup
-------------------------------------
       1     1        0.05     1.00x
       2     2        0.08     0.72x
       4     4        0.11     0.52x
       6     6        0.15     0.36x
      10    10        0.15     0.35x


In [24]:
# 1. unit time
bch_time = time_stage(bch_nb, nb_mib_cells[0])
print(f"BCH unit time: {bch_time:.2f} ms\n")

# 2. concurrency
results = test_concurrency(bch_nb, nb_mib_cells)

print(f"BCH concurrency test: {len(nb_mib_cells)} chunks\n")
print(f"{'workers':>8s}  {'cc':>4s}  {'unit(ms)':>10s}  {'speedup':>8s}")
print("-" * 37)
baseline = results[0][2]
n = len(nb_mib_cells)
for workers, cc, elapsed in results:
    unit_ms = elapsed / n * 1000
    print(f"{workers:>8d}  {cc:>4d}  {unit_ms:>10.2f}  {baseline/elapsed:>7.2f}x")

BCH unit time: 0.14 ms

BCH concurrency test: 100 chunks

 workers    cc    unit(ms)   speedup
-------------------------------------
       1     1        0.21     1.00x
       2     2        0.23     0.92x
       4     4        0.24     0.86x
       6     6        0.28     0.76x
      10    10        0.33     0.63x
